In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/moltbook.db")

# 1 - Distribuzione post per fascia commenti
df1 = pd.read_sql_query("""
SELECT 
  COUNT(*) as post_totali,
  SUM(CASE WHEN comment_count >= 1 THEN 1 ELSE 0 END) as post_con_1plus,
  SUM(CASE WHEN comment_count >= 10 THEN 1 ELSE 0 END) as post_con_10plus,
  AVG(comment_count) as media_commenti
FROM posts;
""", conn)
display(df1)

# 2 - Conteggio agenti nel DB
df2 = pd.read_sql_query("""
  SELECT 'Agenti   ' as tab, COUNT(*) as n FROM agents
  UNION ALL
  SELECT 'Post     ', COUNT(*) FROM posts
  UNION ALL
  SELECT 'Commenti ', COUNT(*) FROM comments
  UNION ALL
  SELECT 'Archi    ', COUNT(*) FROM comments WHERE parent_id IS NOT NULL;
""", conn)
display(df2)

df2 = pd.read_sql_query("""
  SELECT 'Post      ', COUNT(*) FROM posts
  UNION ALL
  SELECT 'Commenti  ', COUNT(*) FROM comments
  UNION ALL
  SELECT 'Archi     ', COUNT(*) FROM comments WHERE parent_id IS NOT NULL
  UNION ALL
  SELECT 'Post autori unici', COUNT(DISTINCT author_name) FROM posts
  UNION ALL
  SELECT 'Agenti mancanti  ', COUNT(DISTINCT p.author_name)
    FROM posts p
    WHERE p.author_name IS NOT NULL
    AND p.author_name NOT IN (SELECT name FROM agents);
""", conn)
display(df2)


df2 = pd.read_sql_query("""
  SELECT
    is_claimed,
    COUNT(*) as n,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct
  FROM agents
  GROUP BY is_claimed;

""", conn)
display(df2)

df_check = pd.read_sql_query("""
SELECT 
  is_claimed,
  COUNT(*) as count
FROM agents
GROUP BY is_claimed;
""", conn)
display(df_check)

df_unclaimed = pd.read_sql_query("""
SELECT
  a.name,
  a.karma,
  a.follower_count,
  COALESCE(written.n, 0) as commenti_scritti,
  COALESCE(received.n, 0) as risposte_ricevute
FROM agents a
LEFT JOIN (
    SELECT author_name, COUNT(*) as n
    FROM comments
    GROUP BY author_name
) written ON written.author_name = a.name
LEFT JOIN (
    SELECT cp.author_name, COUNT(*) as n
    FROM comments cr
    JOIN comments cp ON cr.parent_id = cp.id
    GROUP BY cp.author_name
) received ON received.author_name = a.name
WHERE a.is_claimed = 0
ORDER BY commenti_scritti DESC;
""", conn)

print(f"Unclaimed totali: {len(df_unclaimed)}")
print(f"Con almeno 1 commento: {len(df_unclaimed[df_unclaimed['commenti_scritti'] > 0])}")
print(f"Senza commenti: {len(df_unclaimed[df_unclaimed['commenti_scritti'] == 0])}")
display(df_unclaimed.head(20))


df_unclaimed = pd.read_sql_query("""
SELECT 
  'posts' as tabella,
  MIN(created_at) as data_min,
  MAX(created_at) as data_max,
  COUNT(*) as totale
FROM posts
UNION ALL
SELECT 
  'comments',
  MIN(created_at),
  MAX(created_at),
  COUNT(*)
FROM comments
UNION ALL
SELECT 
  'agents',
  MIN(created_at),
  MAX(created_at),
  COUNT(*)
FROM agents;
""", conn)

display(df_unclaimed.head(20))

conn.close()

,post_totali,post_con_1plus,post_con_10plus,media_commenti
0,311448,211834,20783,4.688792


,tab,n
0,Agenti,27107
1,Post,311448
2,Commenti,794814
3,Archi,191671


,'Post ',COUNT(*)
0,Post,311448
1,Commenti,794814
2,Archi,191671
3,Post autori unici,20774
4,Agenti mancanti,3


,is_claimed,n,pct
0,0,369,1.4
1,1,26738,98.6


,is_claimed,count
0,0,369
1,1,26738
